In [4]:
# 개와 고양이 이진분류 예제를 EfficientNet을 이용해서 구현
# Tensorflow Keras에서는 EfficientNet을 공식적으로 지원하고 있어요!

# EfficientNet은 버전이 여러개가 있어요!
# EfficientNetB0 ~ B7까지 버전이 있어요!
# B0 : 가장 간단한 모델이고 빠르게 학습시킬 수 있어요!
# 일반적으로 B3~B4를 사용. B5이상은 시간이 오래걸리지만 정확도가 높아요!
# EfficientNetV2B0~B3까지 개량된 모델도 있어요!
# 내가 가지고 있는 이미지의 크기와 살짝 연관성이 있어요!
# EfficinetNet 사용할 때 입력이미지의 크기를 고정시켜서 사용하는게 좋아요!
# 우리는 EfficinetNetB4를 사용할거기 때문에 입력이미지의 크기는 
# 380x380으로 설정할거예요!
%reset
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import Dropout, GlobalAveragePooling2D 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import mixed_precision
# float16과 float32를 적절히 섞어서 학습속도를 높이고 GPU의 메모리 사용량을
# 줄일 수 있어요!
mixed_precision.set_global_policy('mixed_float16')
# 이렇게 하면 모든 연산이 float16으로 계산되고
# 맨 마지막 output layer만 float32로 사용

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [6]:
# 입력데이터 처리
train_dir = './data/cat_dog_small/train'
validation_dir = './data/cat_dog_small/validation'

# Parameter 설정
IMAGE_SIZE = 380
BATCH_SIZE = 64   # OOM 오류가 발생하면 숫자를 줄여야 해요!
LEARNING_RATE = 5e-5   # EfficinetNet은 상당히 민감한 모델이예요. learning_rate에 영향을
                       # 많이 받는 모델이예요. 따라서 이 값이 크면 불안정해져요!
                       # 일반적으로 사용하는 값보다 작게 잡는게 좋아요!

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                   rotation_range=40,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   fill_mode='nearest')

validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    classes=['cats', 'dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    classes=['cats', 'dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [10]:
# Model

pretrained_network = EfficientNetB4(weights='imagenet',
                                   include_top=False,
                                   input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))

# pretrained_network.trainable = False  
# 이렇게 하는게 맞긴한데.. 간혹 동결이 풀리는 경우가 있어요!
for layer in pretrained_network.layers:
    layer.trainable = False

model = Sequential()

model.add(pretrained_network)
# model.add(Flatten())  # 12 * 12 * xxx
model.add(GlobalAveragePooling2D())  # 1 * xxx
model.add(Dense(units=64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(units=1,
                activation='sigmoid'))

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='binary_crossentropy',
              metrics=['accuracy'])

cp_callback = ModelCheckpoint(filepath='./EfficientNet_weights.h5',
                              save_best_only=True,
                              save_weights_only=True,
                              monitor='val_accuracy',
                              verbose=1)

es_callback = EarlyStopping(monitor='val_loss',
                            patience=5,
                            verbose=1)

In [ ]:
# 1차 학습
model.fit(train_generator,
          steps_per_epoch=len(train_generator),
          epochs=20,
          validation_data =validation_generator,
          validation_steps=len(validation_generator),
          callbacks=[cp_callback, es_callback],
          verbose=1)

In [ ]:
# Fine Tuning

for layer in pretrained_network.layers:
    layer.trainable = True

for layer in pretrained_network.layers[:,-30]:
    layer.trainable = False

# 반드시 다시 compile작업을 해야 해요!
# Learnign_rate는 줄여서 사용해야 하구요!
# Optimizer(Adam)도 새롭게 생성해서 사용해야 해요!
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE * 0.1),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 재학습(epoch수는 조금 증가시키는게 일반적])
model.fit(train_generator,
          steps_per_epoch=len(train_generator),
          epochs=30,
          validation_data =validation_generator,
          validation_steps=len(validation_generator),
          callbacks=[cp_callback, es_callback],
          verbose=1)